# Multiproduct MLP benchmark

Protocol aligned with ICDN:
- Optuna on 3 expanding folds × 3 seeds, maximize `robust_r2 = mean_r2 - 0.25 * std_r2`
- Final k-fold: 5 expanding folds × 5 seeds
- Block bootstrap on the 80/20 holdout

One-phase Huber MLP. Elasticities = Jacobian. No Phase 0/1, no pair dataset.

In [ ]:
import sys
import json
import random
from pathlib import Path
from dataclasses import replace

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import optuna

from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory
from src.multiproduct import MultiProductDataset
from src.utils import TemporalSplitter, BlockBootstrapSampler
from src.benchmarks import MLPConfig, DemandMLPPipeline

In [ ]:
BASE_SEED = 42

N_UPCS = 5
MIN_TRAIN_FRAC = 0.5
TRAIN_FRAC = 0.8
BLOCK_SIZE = 4

# Tune (ICDN hparam-search)
TUNE_N_FOLDS = 3
TUNE_SEEDS = [11, 29, 42]
N_TRIALS = 100          # smoke: 2

# Final eval (ICDN nn_final_evaluation)
N_FOLDS = 5
EVAL_SEEDS = [11, 29, 42, 77, 123]   # smoke: [42]
N_BOOTSTRAP = 20                    # smoke: 2

HIDDEN_OPTIONS = {
    "64_32":      (64, 32),
    "128_64":     (128, 64),
    "192_96":     (192, 96),
    "256_128":    (256, 128),
    "256_128_64": (256, 128, 64),
}

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_TRIAL_PATH = RESULTS_DIR / "mlp_best_trial_params.json"
TRIAL_SUMMARY_PATH = RESULTS_DIR / "mlp_hparam_trials_summary.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(BASE_SEED)

# Data

In [ ]:
loader = DominickDataLoader()
df = loader.load("elasticity_dataset.csv")

encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id", sort=True)
_, brand_cats = encoder.factorize(df, "brand_family_norm", sort=True)
_, style_cats = encoder.factorize(df, "style_segment_norm", sort=True)

n_stores = len(store_cats)
n_brands = len(brand_cats)
n_styles = len(style_cats)

brand_map = {v: i + 1 for i, v in enumerate(brand_cats)}
style_map = {v: i + 1 for i, v in enumerate(style_cats)}
df["brand_family_norm"] = df["brand_family_norm"].map(brand_map).fillna(0).astype(int)
df["style_segment_norm"] = df["style_segment_norm"].map(style_map).fillna(0).astype(int)

mp_builder = MultiProductBuilder()
mp_builder.fit(df, n_upcs=N_UPCS)
full_wide_raw = mp_builder.transform().copy()
n_upcs = mp_builder.n
upc_names = mp_builder.selected_upcs
n_product_feats = len(MultiProductDataset.PER_PRODUCT_COLS)

store_map = {v: i for i, v in enumerate(store_cats)}
week_map = {v: i for i, v in enumerate(week_cats)}

print(f"Dataset: {df.shape}  wide: {full_wide_raw.shape}")
print(f"Stores: {n_stores}  weeks: {len(week_cats)}  UPCs: {list(upc_names)}")

# Temporal folds

In [ ]:
splitter = TemporalSplitter(week_col="week_id")

tune_splits = splitter.expanding_splits(
    full_wide_raw, n_folds=TUNE_N_FOLDS, min_train_frac=MIN_TRAIN_FRAC,
)
fold_splits = splitter.expanding_splits(
    full_wide_raw, n_folds=N_FOLDS, min_train_frac=MIN_TRAIN_FRAC,
)
train_final, val_final = splitter.single_split(full_wide_raw, train_frac=TRAIN_FRAC)
train_weeks_final = sorted(train_final["week_id"].unique())

print("Tune folds:", len(tune_splits), "  Eval folds:", len(fold_splits))
for i, (tr, va) in enumerate(fold_splits):
    print(
        f"  eval fold {i}: train={len(tr):,} val={len(va):,} "
        f"weeks {tr['week_id'].nunique()}/{va['week_id'].nunique()}"
    )
print(
    f"Holdout: train={len(train_final):,} ({len(train_weeks_final)} weeks) "
    f"val={len(val_final):,}"
)

# Helpers

In [ ]:
def encode_fold(train_wide, val_wide):
    train_wide, val_wide = train_wide.copy(), val_wide.copy()
    for w in (train_wide, val_wide):
        w["store_code"] = w["store_code"].map(store_map)
        w["week_id"] = w["week_id"].map(week_map)
    return train_wide, val_wide


def make_loaders(train_wide, val_wide, batch_size: int, seed: int):
    factory = DataLoaderFactory(num_workers=4, pin_memory=True, persistent_workers=True)
    train_ds = MultiProductDataset(train_wide, n=n_upcs)
    val_ds = MultiProductDataset(val_wide, n=n_upcs)
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = factory.create_train_loader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True, generator=g,
    )
    val_loader = factory.create_eval_loader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader


def config_from_params(params) -> MLPConfig:
    return replace(
        MLPConfig(),
        hidden=HIDDEN_OPTIONS[params["HIDDEN_KEY"]],
        dropout=float(params["DROPOUT"]),
        lr=float(params["LR"]),
        batch_size=int(params["BATCH_SIZE"]),
    )


def make_pipeline(cfg: MLPConfig, seed: int) -> DemandMLPPipeline:
    return DemandMLPPipeline(
        cfg,
        n=n_upcs,
        n_stores=n_stores,
        n_brands=n_brands,
        n_styles=n_styles,
        n_product_feats=n_product_feats,
        device=device,
        seed=seed,
    )


def fit_metrics(train_fold, val_fold, cfg: MLPConfig, seed: int) -> dict:
    """Search path: predictive metrics + ICDN elast_score (no row explosion)."""
    set_all_seeds(seed)
    train_wide, val_wide = encode_fold(train_fold, val_fold)
    train_loader, val_loader = make_loaders(train_wide, val_wide, cfg.batch_size, seed)
    pipe = make_pipeline(cfg, seed)
    pipe.fit(train_loader, val_loader)
    return {**pipe.metrics(val_loader), **pipe.elasticity_score(val_loader)}


def fit_evaluate(train_fold, val_fold, cfg: MLPConfig, seed: int):
    """Eval path: metrics + Jacobian elasticities."""
    set_all_seeds(seed)
    train_wide, val_wide = encode_fold(train_fold, val_fold)
    train_loader, val_loader = make_loaders(train_wide, val_wide, cfg.batch_size, seed)
    pipe = make_pipeline(cfg, seed)
    pipe.fit(train_loader, val_loader)
    return pipe.evaluate(
        val_loader, store_cats=store_cats, upc_names=upc_names, week_cats=week_cats,
    )

# Optuna

In [ ]:
trial_records = []

def objective(trial):
    params = {
        "HIDDEN_KEY": trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS)),
        "DROPOUT": trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR": trial.suggest_float("LR", 1e-4, 1e-2, log=True),
        "BATCH_SIZE": trial.suggest_categorical("BATCH_SIZE", [256, 512, 1024]),
    }
    print(f"\n{'='*70}\nTrial {trial.number}\n{params}\n{'='*70}")
    cfg = config_from_params(params)

    rows = []
    for fold_id, (tr, va) in enumerate(tune_splits):
        for seed in TUNE_SEEDS:
            m = fit_metrics(tr, va, cfg, seed)
            rows.append({"fold": fold_id, "seed": seed, **m})

    df_t = pd.DataFrame(rows)

    mean_r2 = float(df_t["r2_val"].mean())
    std_r2 = float(df_t["r2_val"].std(ddof=1)) if len(df_t) > 1 else 0.0
    mean_elast = float(df_t["elast_score"].mean())
    std_elast = float(df_t["elast_score"].std(ddof=1)) if len(df_t) > 1 else 0.0
    robust_r2 = mean_r2 - 0.25 * std_r2
    robust_elast = mean_elast - 0.25 * std_elast

    trial.set_user_attr("mean_r2", mean_r2)
    trial.set_user_attr("std_r2", std_r2)
    trial.set_user_attr("robust_r2", robust_r2)
    trial.set_user_attr("mean_elast_score", mean_elast)
    trial.set_user_attr("std_elast_score", std_elast)
    trial.set_user_attr("robust_elast", robust_elast)
    trial.set_user_attr("mean_mae", float(df_t["mae_val"].mean()))
    trial.set_user_attr("mean_rmse", float(df_t["rmse_val"].mean()))

    df_t["trial"] = trial.number
    for k, v in params.items():
        df_t[k] = v
    trial_records.extend(df_t.to_dict("records"))
    print(
        f"Trial {trial.number} summary | "
        f"mean_R2={mean_r2:.4f} std_R2={std_r2:.4f} "
        f"robust_R2={robust_r2:.4f} | "
        f"mean_Elast_Score={mean_elast:.4f} std_Elast_Score={std_elast:.4f} "
        f"robust_Elast_Score={robust_elast:.4f} | "
        f"S_select={robust_r2 + robust_elast:.4f}"
    )
    return robust_r2, robust_elast

In [ ]:
study = optuna.create_study(
    directions=["maximize", "maximize"],
    study_name="mlp_hparam_pareto_kfold_seed",
    storage="sqlite:///../results/mlp_hparam_pareto_kfold_seed.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS)
print("Trials:", len(study.trials))

In [ ]:
df_trials_summary = pd.DataFrame([
    {
        "trial": t.number,
        "robust_r2": t.user_attrs.get("robust_r2", np.nan),
        "robust_elast": t.user_attrs.get("robust_elast", np.nan),
        "mean_r2": t.user_attrs.get("mean_r2", np.nan),
        "std_r2": t.user_attrs.get("std_r2", np.nan),
        "mean_elast_score": t.user_attrs.get("mean_elast_score", np.nan),
        "std_elast_score": t.user_attrs.get("std_elast_score", np.nan),
        "mean_mae": t.user_attrs.get("mean_mae", np.nan),
        "mean_rmse": t.user_attrs.get("mean_rmse", np.nan),
        **t.params,
    }
    for t in study.trials if t.values is not None
])
df_trials_summary["robust_score"] = (
    df_trials_summary["robust_r2"].fillna(0.0)
    + df_trials_summary["robust_elast"].fillna(0.0)
)
best_row = df_trials_summary.sort_values("robust_score", ascending=False).iloc[0]
best_trial_payload = {
    "trial": int(best_row["trial"]),
    "robust_score": float(best_row["robust_score"]),
    "mean_r2": float(best_row["mean_r2"]),
    "std_r2": float(best_row["std_r2"]),
    "mean_elast_score": float(best_row["mean_elast_score"]),
    "std_elast_score": float(best_row["std_elast_score"]),
    "params": { ... },
}
with open(BEST_TRIAL_PATH, "w", encoding="utf-8") as f:
    json.dump(best_trial_payload, f, indent=2, ensure_ascii=False)
df_trials_summary.to_csv(TRIAL_SUMMARY_PATH, index=False)
print(json.dumps(best_trial_payload, indent=2))

# K-folds

In [ ]:
with open(BEST_TRIAL_PATH, encoding="utf-8") as f:
    best_trial = json.load(f)
mlp_config = config_from_params(best_trial["params"])
print("Eval config:", mlp_config)

fold_metrics_rows = []
fold_elasticity_rows = []

for fold_id, (train_fold, val_fold) in enumerate(fold_splits):
    for seed in EVAL_SEEDS:
        print(f"=== Fold {fold_id} | Seed {seed} ===")
        metrics, elas = fit_evaluate(train_fold, val_fold, mlp_config, seed)
        fold_metrics_rows.append({
            "run_type": "kfold",
            "fold": fold_id,
            "seed": seed,
            "n_train": len(train_fold),
            "n_val": len(val_fold),
            **metrics,
        })
        elas = elas.copy()
        elas["run_type"] = "kfold"
        elas["run_id"] = f"fold{fold_id}_seed{seed}"
        elas["fold"] = fold_id
        elas["seed"] = seed
        elas["bootstrap_run"] = np.nan
        fold_elasticity_rows.append(elas)
        print("  mae/rmse/r2", metrics["mae_val"], metrics["rmse_val"], metrics["r2_val"])

mlp_kfold_metrics_raw = pd.DataFrame(fold_metrics_rows)
mlp_kfold_elasticities_raw = pd.concat(fold_elasticity_rows, ignore_index=True)
print(mlp_kfold_metrics_raw)
print(mlp_kfold_elasticities_raw["type"].value_counts())

# Bootstrap

In [ ]:
bootstrap_sampler = BlockBootstrapSampler(
    week_col="week_id",
    block_size=BLOCK_SIZE,
    rng=np.random.default_rng(BASE_SEED),
)
BOOTSTRAP_SEED = EVAL_SEEDS[0]

bootstrap_metrics_rows = []
bootstrap_elasticity_rows = []

for b in range(N_BOOTSTRAP):
    print(f"=== Bootstrap {b + 1}/{N_BOOTSTRAP} ===")
    train_bs = bootstrap_sampler.sample(train_final, train_weeks_final)
    metrics, elas = fit_evaluate(train_bs, val_final, mlp_config, BOOTSTRAP_SEED)
    bootstrap_metrics_rows.append({
        "run_type": "bootstrap",
        "bootstrap_run": b,
        "seed": BOOTSTRAP_SEED,
        "n_train": len(train_bs),
        "n_val": len(val_final),
        **metrics,
    })
    elas = elas.copy()
    elas["run_type"] = "bootstrap"
    elas["run_id"] = f"bootstrap{b}"
    elas["fold"] = np.nan
    elas["seed"] = BOOTSTRAP_SEED
    elas["bootstrap_run"] = b
    bootstrap_elasticity_rows.append(elas)
    print("  mae/rmse/r2", metrics["mae_val"], metrics["rmse_val"], metrics["r2_val"])

mlp_bootstrap_metrics_raw = pd.DataFrame(bootstrap_metrics_rows)
mlp_bootstrap_elasticities_raw = pd.concat(bootstrap_elasticity_rows, ignore_index=True)
print(mlp_bootstrap_metrics_raw.head())

# Save

In [ ]:
mlp_kfold_metrics_raw.to_csv(DATA_DIR / "benchmark_mlp_kfold_metrics_raw.csv", index=False)
mlp_kfold_elasticities_raw.to_csv(DATA_DIR / "benchmark_mlp_kfold_raw.csv", index=False)
mlp_bootstrap_metrics_raw.to_csv(DATA_DIR / "benchmark_mlp_bootstrap_metrics_raw.csv", index=False)
mlp_bootstrap_elasticities_raw.to_csv(DATA_DIR / "benchmark_mlp_bootstrap_raw.csv", index=False)
print("Saved MLP k-fold + bootstrap")